# Chapter 26: Splitting Data Properly

        **Part IV - Feeding Data** · Companion notebook for *PyTorch From Ground Up, Volume I: Foundations*

        This notebook follows the chapter's progression with fresh, CPU-friendly examples. Type, change, break, and repair the code; the goal is fluency, not passive reading.

        ## Learning objectives

        - Create honest train, validation, and test splits
- Prevent preprocessing leakage
- Make split membership reproducible


In [ ]:
import torch

torch.manual_seed(42)
print("PyTorch:", torch.__version__)
print("Default device: cpu")


## Reproducible random splits

Use a dedicated seeded Generator so split membership stays stable while other randomness can change.


In [ ]:
from torch.utils.data import TensorDataset, random_split

x = torch.randn(100, 5)
y = (x[:, 0] > 0).long()
dataset = TensorDataset(x, y)
train_set, val_set, test_set = random_split(
    dataset, [70, 15, 15], generator=torch.Generator().manual_seed(42)
)
print(len(train_set), len(val_set), len(test_set))
print("disjoint:", set(train_set.indices).isdisjoint(val_set.indices))


## Fit preprocessing on training only

Validation and test data must not influence means, standard deviations, vocabularies, or feature selection.


In [ ]:
train_x = x[train_set.indices]
mean = train_x.mean(dim=0, keepdim=True)
std = train_x.std(dim=0, keepdim=True).clamp_min(1e-6)
x_standardized = (x - mean) / std
print("training mean after transform:", x_standardized[train_set.indices].mean(dim=0).round(decimals=4))
print("validation remains independent:", x_standardized[val_set.indices].shape)


## A temporal split

Random splitting is wrong when future observations must remain unseen. Sort by time and cut without shuffling.


In [ ]:
times = torch.arange(20)
train_times = times[:12]
val_times = times[12:16]
test_times = times[16:]
print("latest train:", train_times.max().item(), "earliest test:", test_times.min().item())
assert train_times.max() < val_times.min() < test_times.min()


## Practice

        Work through these without looking back first.

        1. Create an 80/10/10 split for 250 samples.
2. Demonstrate leakage by comparing global and train-only means.
3. Describe the right split for data collected over time.

        <details><summary>Study habit</summary>

        Predict every output shape before running a cell. When something fails, read the final line of the error and print the shape, dtype, and device of every tensor involved.

        </details>


In [ ]:
# Your practice space. Replace `pass` with your solution.
pass


## Recap

You completed Chapter 26's companion lab. Before moving on, explain the central idea aloud and recreate the smallest useful example from memory.

**Next:** return to the book for the full explanation, visual reasoning, watch-outs, and chapter exercises.
